## Lesson 2: Zero-Shot Variant Effect Prediction
### What you'll learn
- Use a pLM's MASKED LANGUAGE MODELLING head to score mutations.
- The "wild-type marginal" trick — comparing P(mutant aa) vs P(wild-type aa).
- Why pLMs work as zero-shot variant predictors *without any training*.

### The intuition
ESM-2 was trained by hiding (masking) random amino acids in real protein
sequences and asking the model to fill them back in. To do this well, it
had to learn what amino acids are "plausible" at every position, given the
surrounding context. That's evolutionary / structural intuition, baked in.

So: if a mutation is deleterious, the model will think the wild-type residue
is much more likely than the mutant residue at that position. The score is:

    score(A -> B at position i) = log P(B | sequence with i masked)
                                  - log P(A | sequence with i masked)

  - score < 0  =>  model favours wild-type, mutation is probably deleterious
  - score ~ 0  =>  mutation is neutral
  - score > 0  =>  model thinks mutant is plausible (often tolerated)

This is exactly how pLMs are used in benchmarks like ProteinGym — no training,
just clever use of the masked-LM head.

> **Run order matters.** The cells below build on each other. Run them **top to bottom** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit `NameError: name 'torch' is not defined` (or similar), you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports every library and defines the module-level constants the rest of the notebook relies on.

In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
WILD_TYPE = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

### `score_mutations` (function)

Score a list of single-point mutations using masked-LM scoring.

Args:
    wild_type: the wild-type protein sequence (string).
    mutations: list of (position, mutant_aa) tuples. Position is 0-indexed
               within `wild_type`. mutant_aa is a single-letter amino acid.

Returns:
    list of float log-likelihood-ratio scores, one per mutation.

In [2]:
def score_mutations(wild_type, mutations, model, tokenizer, device):
    # Tokenize the wild-type sequence ONCE, then mutate the token IDs in place.
    inputs = tokenizer(wild_type, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]  # shape (1, L+2) — has <cls> and <eos>

    scores = []
    for pos, mut_aa in mutations:
        # The tokenizer prepends a <cls> token at index 0, so amino-acid
        # position `pos` in the original sequence is at token index `pos + 1`.
        token_idx = pos + 1
        wt_aa = wild_type[pos]

        # Replace the residue at `token_idx` with the <mask> token.
        masked = input_ids.clone()
        masked[0, token_idx] = tokenizer.mask_token_id

        # Forward pass through the masked-LM head. The model returns logits
        # over the FULL vocabulary at every position.
        with torch.no_grad():
            logits = model(masked).logits  # shape (1, L+2, vocab_size)

        # Convert logits at the masked position into log-probabilities.
        log_probs = torch.log_softmax(logits[0, token_idx], dim=-1)

        # Look up the token IDs for the wild-type and mutant amino acids.
        wt_id = tokenizer.convert_tokens_to_ids(wt_aa)
        mut_id = tokenizer.convert_tokens_to_ids(mut_aa)

        # Score = log P(mut) - log P(wt) at the masked position.
        score = (log_probs[mut_id] - log_probs[wt_id]).item()
        scores.append(score)

    return scores

### `saturation_mutagenesis` (function)

Compute scores for EVERY amino acid at EVERY position (up to a limit).

Returns a (positions, 20) numpy matrix of scores.

In [3]:
def saturation_mutagenesis(wild_type, model, tokenizer, device, max_positions=20):
    positions = list(range(min(max_positions, len(wild_type))))
    mutations = [(p, aa) for p in positions for aa in AMINO_ACIDS]
    flat = score_mutations(wild_type, mutations, model, tokenizer, device)
    return np.array(flat).reshape(len(positions), len(AMINO_ACIDS))

### `main` (function)

In [ ]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load the pLM with the MASKED-LM head exposed (not just the encoder body).
    # AutoModelForMaskedLM gives us access to .logits over the vocab.
    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()

    print(f"\nWild-type sequence (length {len(WILD_TYPE)}):")
    print(f"  {WILD_TYPE}")

    # ---- Demo 1: a few hand-picked mutations -----------------------------
    # Convention: <wt><pos+1><mut>. e.g. "M1A" = position 0 (M) -> A.
    test_mutations = [
        (0, "A"),    # M1A   - changes start codon, usually catastrophic
        (1, "L"),    # K2L   - charged -> hydrophobic, often disruptive
        (5, "A"),    # Q6A   - alanine scan: removes side-chain function
        (10, "K"),   # E11K  - charge swap (acidic -> basic)
        (10, "D"),   # E11D  - conservative (acidic -> acidic)
    ]

    print("\nScoring single mutations...")
    scores = score_mutations(WILD_TYPE, test_mutations, model, tokenizer, device)

    print(f"\n{'Mutation':<10}{'Score':>10}    Interpretation")
    print("-" * 50)
    for (pos, mut_aa), s in zip(test_mutations, scores):
        wt_aa = WILD_TYPE[pos]
        mut_str = f"{wt_aa}{pos + 1}{mut_aa}"
        if s > 0:
            verdict = "model prefers mutant (likely tolerated)"
        elif s > -2:
            verdict = "neutral / mildly disfavoured"
        else:
            verdict = "model strongly prefers wild-type (likely deleterious)"
        print(f"{mut_str:<10}{s:>+10.3f}    {verdict}")

    # ---- Demo 2: full saturation mutagenesis on a window -----------------
    # For each position, score every possible substitution. This gives you
    # a (positions, 20) heatmap — exactly what DMS experiments measure.
    print("\nRunning saturation mutagenesis on first 20 positions...")
    matrix = saturation_mutagenesis(WILD_TYPE, model, tokenizer, device, max_positions=20)

    print(f"  Score matrix shape: {matrix.shape}  (positions, amino_acids)")
    print(f"  Mean score:    {matrix.mean():+.3f}  (typically negative — mutations usually disfavoured)")
    print(f"  Min  / Max:    {matrix.min():+.3f} / {matrix.max():+.3f}")

    # Show a small ASCII preview: which AA does the model prefer at each position?
    print("\nTop preferred AA per position (vs wild-type letter):")
    for p in range(min(20, len(WILD_TYPE))):
        wt_aa = WILD_TYPE[p]
        best_idx = matrix[p].argmax()
        best_aa = AMINO_ACIDS[best_idx]
        same = " (matches WT)" if best_aa == wt_aa else ""
        print(f"  pos {p + 1:3d}  WT={wt_aa}  model_prefers={best_aa}{same}")

    print(
        """
Things to experiment with:
- Use ESM-1v (purpose-built for variant prediction):
    MODEL_NAME = "facebook/esm1v_t33_650M_UR90S_1"   # 650M, GPU recommended
- The three scoring schemes (masked / wild-type marginal, pseudo-likelihood) and a
  real ProteinGym DMS benchmark with Spearman correlation are implemented in the
  sections below this one.
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [5]:
main()

Using device: cuda
Loading model: facebook/esm2_t6_8M_UR50D


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]


Wild-type sequence (length 65):
  MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG

Scoring single mutations...

Mutation       Score    Interpretation
--------------------------------------------------
M1A           -7.370    model strongly prefers wild-type (likely deleterious)
K2L           -0.667    neutral / mildly disfavoured
Q6A           +0.586    model prefers mutant (likely tolerated)
S11K          +1.024    model prefers mutant (likely tolerated)
S11D          -0.223    neutral / mildly disfavoured

Running saturation mutagenesis on first 20 positions...


  Score matrix shape: (20, 20)  (positions, amino_acids)
  Mean score:    -2.105  (typically negative — mutations usually disfavoured)
  Min  / Max:    -9.702 / +2.290

Top preferred AA per position (vs wild-type letter):
  pos   1  WT=M  model_prefers=M (matches WT)
  pos   2  WT=K  model_prefers=S
  pos   3  WT=T  model_prefers=T (matches WT)
  pos   4  WT=V  model_prefers=S
  pos   5  WT=R  model_prefers=R (matches WT)
  pos   6  WT=Q  model_prefers=S
  pos   7  WT=E  model_prefers=E (matches WT)
  pos   8  WT=R  model_prefers=R (matches WT)
  pos   9  WT=L  model_prefers=L (matches WT)
  pos  10  WT=K  model_prefers=L
  pos  11  WT=S  model_prefers=R
  pos  12  WT=I  model_prefers=I (matches WT)
  pos  13  WT=V  model_prefers=L
  pos  14  WT=R  model_prefers=E
  pos  15  WT=I  model_prefers=L
  pos  16  WT=L  model_prefers=L (matches WT)
  pos  17  WT=E  model_prefers=L
  pos  18  WT=R  model_prefers=E
  pos  19  WT=S  model_prefers=E
  pos  20  WT=K  model_prefers=G

Things to exp

## Scoring schemes: masked-marginal vs wild-type-marginal vs pseudo-likelihood

The lesson above used the **masked-marginal** score. It isn't the only way to turn a
masked-LM into a variant scorer — the ESM-1v paper compares three, trading accuracy for
compute:

- **masked marginal** — mask each scored position, read log-probs there. One forward per
  position. (This lesson's approach.)
- **wild-type marginal** — never mask; a single forward on the WT sequence, read log-probs
  at each position. Cheapest (one forward total), slightly less accurate.
- **pseudo-likelihood (PLL)** — `score = PLL(mutant) − PLL(wt)`, where `PLL` sums masked
  log-probs over *all* positions. Most context-aware, but **O(L) forwards per variant**.

All three follow the same convention: `score > 0 ⇒ model favours the mutant`. We implement
all three, compare them on the demo mutations, then benchmark against real DMS data.

In [ ]:
# Load the masked-LM once for the sections below and define the three scorers.
# Each scorer takes (wt_seq, variants) where a variant is
# (pos0, wt_aa, mut_aa, *rest) with pos0 a 0-based index into wt_seq.
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()

def _token_logits(input_ids):
    with torch.no_grad():
        return mlm_model(input_ids.to(device)).logits[0]   # (L+2, vocab)

def _aid(aa):
    return tokenizer.convert_tokens_to_ids(aa)

def masked_marginal_scores(wt_seq, variants):
    """score = logP(mut) - logP(wt) at the MASKED position. One forward per
    distinct position (cached), so cost ~ number of distinct positions scored."""
    base = tokenizer(wt_seq, return_tensors="pt")["input_ids"]
    cache, out = {}, []
    for pos0, wt_aa, mut_aa, *_ in variants:
        t = pos0 + 1                       # +1 skips the <cls> token
        if t not in cache:
            masked = base.clone(); masked[0, t] = tokenizer.mask_token_id
            cache[t] = torch.log_softmax(_token_logits(masked)[t], dim=-1)
        lp = cache[t]
        out.append((lp[_aid(mut_aa)] - lp[_aid(wt_aa)]).item())
    return out

def wt_marginal_scores(wt_seq, variants):
    """ESM-1v 'wild-type marginal': never mask. A single forward on the WT
    sequence, read log-probs at each position. Cheapest scheme (one forward)."""
    base = tokenizer(wt_seq, return_tensors="pt")["input_ids"]
    lp_all = torch.log_softmax(_token_logits(base), dim=-1)
    out = []
    for pos0, wt_aa, mut_aa, *_ in variants:
        lp = lp_all[pos0 + 1]
        out.append((lp[_aid(mut_aa)] - lp[_aid(wt_aa)]).item())
    return out

def pseudo_likelihood_scores(wt_seq, variants, max_scored=None):
    """score = PLL(mutant) - PLL(wt), with PLL(seq) = sum_i logP(x_i | x_{\\i} masked).
    Most context-aware, but O(L) forwards PER variant -- pass max_scored to only
    compute the first N variants (the rest come back as nan)."""
    base = tokenizer(wt_seq, return_tensors="pt")["input_ids"]
    L = base.shape[1]
    def pll(ids):
        total = 0.0
        for t in range(1, L - 1):          # skip <cls>/<eos>
            masked = ids.clone(); masked[0, t] = tokenizer.mask_token_id
            lp = torch.log_softmax(_token_logits(masked)[t], dim=-1)
            total += lp[ids[0, t]].item()
        return total
    pll_wt = pll(base)                      # shared across all variants
    out = []
    for i, (pos0, wt_aa, mut_aa, *_) in enumerate(variants):
        if max_scored is not None and i >= max_scored:
            out.append(float("nan")); continue
        mut_ids = base.clone(); mut_ids[0, pos0 + 1] = _aid(mut_aa)
        out.append(pll(mut_ids) - pll_wt)
    return out

print(f"Scorers ready on {device}. Model: {MODEL_NAME}")

In [ ]:
# Compare all three schemes on the same hand-picked mutations from Demo 1.
demo_muts = [(0, "A"), (1, "L"), (5, "A"), (10, "K"), (10, "D")]
demo_variants = [(p, WILD_TYPE[p], a, None) for p, a in demo_muts]

mm  = masked_marginal_scores(WILD_TYPE, demo_variants)
wm  = wt_marginal_scores(WILD_TYPE, demo_variants)
pll = pseudo_likelihood_scores(WILD_TYPE, demo_variants)   # short seq -> affordable

print(f"{'Mutation':<10}{'masked':>10}{'wt':>10}{'pseudoLL':>12}")
print("-" * 42)
for (p, wt_aa, mut_aa, _), a, b, c in zip(demo_variants, mm, wm, pll):
    print(f"{wt_aa}{p + 1}{mut_aa:<6}{a:>+10.3f}{b:>+10.3f}{c:>+12.3f}")

print("\nThe three schemes usually AGREE in sign; they differ in magnitude/sensitivity.")
print("masked & wt marginal are cheap; pseudo-likelihood costs O(L) forwards per variant.")

## Benchmark against a real DMS dataset (ProteinGym)

The real test of a zero-shot scorer is whether it tracks **measured** fitness. We pull a
single Deep Mutational Scanning (DMS) assay from [ProteinGym](https://proteingym.org/) via
the `plm_common.load_dms_assay` helper (streamed from the HuggingFace Hub, with a clearly
labelled synthetic fallback if you're offline), score every single-substitution variant
with each scheme, and report **Spearman correlation** against the experimental `DMS_score`.

> ESM-2 8M is tiny, so expect modest correlations. Purpose-built models (ESM-1v, larger
> ESM-2) score much higher on ProteinGym — the point here is the *method*, not the number.
> The first run downloads from the Hub; pass `assay_id="..."` to pick a specific study.

In [ ]:
# Import the track-local DMS loader (plm/plm_common.py).
import os, sys
for _cand in (os.path.abspath(""), os.path.join(os.path.abspath(""), "plm")):
    if os.path.isfile(os.path.join(_cand, "plm_common.py")) and _cand not in sys.path:
        sys.path.insert(0, _cand)
import plm_common

# Pull one real ProteinGym assay; fall back to a clearly-labelled synthetic stub
# if the Hub is unreachable. Pass assay_id="PABP_YEAST_Melamed_2013" (etc.) to
# pick a specific study; default takes the first assay in the stream.
try:
    assay, wt_seq, variants = plm_common.load_dms_assay(max_variants=150)
    real = True
    print(f"Loaded ProteinGym assay: {assay}")
except Exception as e:
    assay, wt_seq, variants = plm_common.fallback_dms_assay()
    real = False
    print(f"[fallback] Could not load ProteinGym ({e}).")
    print("Using a SYNTHETIC stub -- the correlations below are meaningless.")

# ESM-2's positional limit is ~1024; truncate very long references (and drop
# variants past the cut) so the single-forward schemes don't overflow.
MAXLEN = 1022
if len(wt_seq) > MAXLEN:
    wt_seq = wt_seq[:MAXLEN]
    variants = [v for v in variants if v[0] < MAXLEN]

print(f"  WT length: {len(wt_seq)},  single-substitution variants: {len(variants)}")
y = [v[3] for v in variants]   # experimental DMS_score (higher = fitter)

# Cheap schemes over all variants; expensive PLL only on the first PLL_CAP.
s_masked = masked_marginal_scores(wt_seq, variants)
s_wt     = wt_marginal_scores(wt_seq, variants)
PLL_CAP  = 40
s_pll    = pseudo_likelihood_scores(wt_seq, variants, max_scored=min(PLL_CAP, len(variants)))

def _spearman(scores):
    sc, yy = np.array(scores, float), np.array(y, float)
    ok = ~np.isnan(sc)
    if ok.sum() < 3:
        return float("nan"), int(ok.sum())
    return float(spearmanr(sc[ok], yy[ok]).correlation), int(ok.sum())

print(f"\n{'Scheme':<26}{'Spearman r':>12}{'n':>7}")
print("-" * 45)
for name, sc in [("masked marginal", s_masked),
                 ("wild-type marginal", s_wt),
                 (f"pseudo-likelihood (n<={PLL_CAP})", s_pll)]:
    r, n = _spearman(sc)
    print(f"{name:<26}{r:>+12.3f}{n:>7}")

print("\nHigher (more positive) Spearman => zero-shot scores track measured fitness better.")
if not real:
    print("NOTE: synthetic fallback data -> ignore these numbers; re-run online for the real benchmark.")
print("ESM-1v / larger ESM-2 lift these correlations substantially.")